# Adsorptive Desulfurization Sorbents: Capacity Modeling
## 2. Feature Engineering

This notebook builds model-ready feature tables for `capacity_mgS_g` (n=36 after the exclusion described below), continuing from `01_eda.ipynb`. Zeolites and activated carbons are engineered and modeled separately.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/adsorptive_desulfurization_dataset.csv')
cap = df[df['capacity_mgS_g'].notna()].copy()
ac = cap[cap['sorbent_class'] == 'activated_carbon'].copy()
zeo = cap[cap['sorbent_class'] != 'activated_carbon'].copy()

# 2Cu10CeSAY is excluded: it is the only zeolite tested with DMDBT, has the highest RE and Cu
# loadings among the bimetallic zeolites, and comes from a different source than the rest of the
# RE-loading series, so these factors are confounded in a single observation
# (see 01_eda.ipynb, section 1b).
zeo = zeo[zeo['material'] != '2Cu10CeSAY'].copy()

print(f"Activated carbon rows: {len(ac)}, Zeolite rows: {len(zeo)}")

Activated carbon rows: 20, Zeolite rows: 16


## Part A: Zeolite features (n=16)

### A.1 Textural features

In [2]:
zeo.groupby('sulfur_compound')['Vmeso_cm3g'].agg(['min', 'max'])


,min,max
sulfur_compound,,
BT,0.043,0.056
DBT,0.160,0.243


In [3]:
zeolite_features = zeo[['material', 'RE_type', 'RE_wt_pct', 'Cu_wt_pct',
                         'Vmicro_cm3g', 'Vmeso_cm3g', 'sulfur_compound', 'capacity_mgS_g']].copy()
zeolite_features


,material,RE_type,RE_wt_pct,Cu_wt_pct,Vmicro_cm3g,Vmeso_cm3g,sulfur_compound,capacity_mgS_g
20,LaY,La,4.9,0.0,0.262,0.043,BT,0.64
21,NdY,Nd,5.0,0.0,0.255,0.048,BT,0.60
22,SmY,Sm,4.7,0.0,0.265,0.051,BT,0.50
23,CeY,Ce,5.1,0.0,0.259,0.050,BT,0.84
24,CuLaY,La,2.3,2.5,0.280,0.049,BT,1.23
25,CuNdY,Nd,2.6,2.8,0.271,0.056,BT,1.40
26,CuSmY,Sm,2.4,2.5,0.271,0.049,BT,0.85
27,CuCeY,Ce,2.5,2.2,0.267,0.053,BT,1.76
28,LaSAY,La,3.5,0.0,0.184,0.178,DBT,1.00
29,NdSAY,Nd,3.8,0.0,0.166,0.243,DBT,0.90


### A.2 Metal and compound identity

Each rare earth is encoded by the 4f electron count of its 3+ ion, the state present immediately before sulfur adsorption (all four are trivalent after reduction, including Ce): La = 0, Ce = 1, Nd = 3, Sm = 5. These are standard electron configurations, not per-material measurements.

Capacity peaked at Ce (4f1) in every group (see `01_eda.ipynb`), a pattern that a linear term in `RE_f_electrons` cannot represent. Two derived columns are added:

- `RE_f_dist_from_1` = |f electrons - 1|: 0 for Ce, 1 for La, 2 for Nd and 4 for Sm.
- `Cu_x_f_dist` = `Cu_wt_pct` x `RE_f_dist_from_1`: the Cu-Ce electronic interaction reported in the DFT work (Cu 3d donation into Ce 4f/5d) should be relevant only where Cu is present.

These two columns were designed after the Ce peak was observed in the exploratory analysis, so comparisons involving them are exploratory. With four rare earths, both columns are recodings of rare-earth identity. They allow the models to use the Ce peak but do not establish that the 4f1 configuration is its cause.

The table below contains every candidate column: `RE_f_electrons`, `RE_f_dist_from_1`, `Cu_x_f_dist`, `RE_wt_pct`, `Cu_wt_pct`, `Vmicro_cm3g`, `Vmeso_cm3g` and two compound indicators (BT, DBT; the DMDBT row was excluded above). `compound_BT` and `compound_DBT` are exact complements (and also identify Y versus SAY), so the nine columns contain eight independent features. `03_modeling.ipynb` compares the original seven columns with the full nine on the same 16 rows.

In [4]:
re_f_electrons_map = {'La': 0, 'Ce': 1, 'Nd': 3, 'Sm': 5}
zeolite_features['RE_f_electrons'] = zeolite_features['RE_type'].map(re_f_electrons_map)
zeolite_features['RE_f_dist_from_1'] = (zeolite_features['RE_f_electrons'] - 1).abs()
zeolite_features['Cu_x_f_dist'] = zeolite_features['Cu_wt_pct'] * zeolite_features['RE_f_dist_from_1']
zeolite_features_encoded = zeolite_features.drop(columns=['RE_type'])

zeolite_model_table = pd.get_dummies(zeolite_features_encoded, columns=['sulfur_compound'],
                                      prefix='compound')
dummy_cols = [c for c in zeolite_model_table.columns if c.startswith('compound_')]
zeolite_model_table[dummy_cols] = zeolite_model_table[dummy_cols].astype(int)
zeolite_model_table.to_csv('../data/zeolite_features.csv', index=False)
print(zeolite_model_table.shape)
zeolite_model_table

(16, 11)


,material,RE_wt_pct,Cu_wt_pct,Vmicro_cm3g,Vmeso_cm3g,capacity_mgS_g,RE_f_electrons,RE_f_dist_from_1,Cu_x_f_dist,compound_BT,compound_DBT
20,LaY,4.9,0.0,0.262,0.043,0.64,0,1,0.0,1,0
21,NdY,5.0,0.0,0.255,0.048,0.60,3,2,0.0,1,0
22,SmY,4.7,0.0,0.265,0.051,0.50,5,4,0.0,1,0
23,CeY,5.1,0.0,0.259,0.050,0.84,1,0,0.0,1,0
24,CuLaY,2.3,2.5,0.280,0.049,1.23,0,1,2.5,1,0
25,CuNdY,2.6,2.8,0.271,0.056,1.40,3,2,5.6,1,0
26,CuSmY,2.4,2.5,0.271,0.049,0.85,5,4,10.0,1,0
27,CuCeY,2.5,2.2,0.267,0.053,1.76,1,0,0.0,1,0
28,LaSAY,3.5,0.0,0.184,0.178,1.00,0,1,0.0,0,1
29,NdSAY,3.8,0.0,0.166,0.243,0.90,3,2,0.0,0,1


## Part B: Activated carbon features (n=20)

### B.1 Constant columns

`co_adsorbate`, `feed_conc_ppmw` and `solvent` have zero variance within the activated-carbon subset and are dropped.

In [5]:
print(ac['co_adsorbate'].unique(), ac['feed_conc_ppmw'].unique(), ac['solvent'].unique())

<StringArray>
['naphthalene_1wt%']
Length: 1, dtype: str [100] <StringArray>
['n-octane']
Length: 1, dtype: str


### B.2 Inorganic composition

In [6]:
ac[['material','inorganic_P_wt_pct','inorganic_Ca_wt_pct','inorganic_K_wt_pct','inorganic_Na_wt_pct']]


,material,inorganic_P_wt_pct,inorganic_Ca_wt_pct,inorganic_K_wt_pct,inorganic_Na_wt_pct
36,MisAC,1.0,1.0,3.5,0.2
37,MisAC,1.0,1.0,3.5,0.2
38,WAC,0.1,0.8,1.7,0.0
39,WAC,0.1,0.8,1.7,0.0
40,CoAC,0.3,0.1,2.0,0.3
41,CoAC,0.3,0.1,2.0,0.3
42,CAC,1.7,1.7,0.0,0.6
43,CAC,1.7,1.7,0.0,0.6
44,FWAC65,1.0,0.7,0.8,0.7
45,FWAC65,1.0,0.7,0.8,0.7


### B.3 Textural, compositional and Raman features

The features are the micro and meso surface areas and pore volumes, the full elemental analysis (inorganic P, Ca, K, Na and C, H, N, O) and `ID_IG_ratio` (Raman D/G band ratio, a measure of carbon structural disorder), which had the strongest association with capacity in `01_eda.ipynb`. Total surface area and total pore volume are omitted because they are the sums of the micro and meso values.

In [7]:
ac_features = ac[['material', 'sulfur_compound', 'BET_micro_m2g', 'BET_meso_m2g', 'Vmicro_cm3g', 'Vmeso_cm3g',
                   'inorganic_P_wt_pct', 'inorganic_Ca_wt_pct', 'inorganic_K_wt_pct', 'inorganic_Na_wt_pct',
                   'carbon_C_wt_pct', 'carbon_H_wt_pct', 'carbon_N_wt_pct', 'oxygen_O_wt_pct',
                   'ID_IG_ratio', 'capacity_mgS_g']].copy()
ac_features

,material,sulfur_compound,BET_micro_m2g,BET_meso_m2g,Vmicro_cm3g,Vmeso_cm3g,inorganic_P_wt_pct,inorganic_Ca_wt_pct,inorganic_K_wt_pct,inorganic_Na_wt_pct,carbon_C_wt_pct,carbon_H_wt_pct,carbon_N_wt_pct,oxygen_O_wt_pct,ID_IG_ratio,capacity_mgS_g
36,MisAC,DBT,563.0,308.0,0.259,0.304,1.0,1.0,3.5,0.2,63.4,1.8,0.2,28.9,1.9,4.70
37,MisAC,DMDBT,563.0,308.0,0.259,0.304,1.0,1.0,3.5,0.2,63.4,1.8,0.2,28.9,1.9,9.10
38,WAC,DBT,634.0,295.0,0.293,0.208,0.1,0.8,1.7,0.0,88.4,1.1,0.1,7.8,1.8,4.20
39,WAC,DMDBT,634.0,295.0,0.293,0.208,0.1,0.8,1.7,0.0,88.4,1.1,0.1,7.8,1.8,7.40
40,CoAC,DBT,614.0,240.0,0.283,0.199,0.3,0.1,2.0,0.3,87.1,1.0,0.5,8.7,2.0,5.10
41,CoAC,DMDBT,614.0,240.0,0.283,0.199,0.3,0.1,2.0,0.3,87.1,1.0,0.5,8.7,2.0,7.00
42,CAC,DBT,351.0,567.0,0.157,0.613,1.7,1.7,0.0,0.6,55.8,2.8,0.2,37.2,1.1,2.30
43,CAC,DMDBT,351.0,567.0,0.157,0.613,1.7,1.7,0.0,0.6,55.8,2.8,0.2,37.2,1.1,2.00
44,FWAC65,DBT,329.0,151.0,0.153,0.163,1.0,0.7,0.8,0.7,75.0,1.5,1.7,18.6,2.1,3.10
45,FWAC65,DMDBT,329.0,151.0,0.153,0.163,1.0,0.7,0.8,0.7,75.0,1.5,1.7,18.6,2.1,3.80


### B.4 Sulfur compound as a binary indicator

DMDBT capacity exceeded DBT capacity for 9 of the 10 materials tested with both (see `01_eda.ipynb`), so the sulfur compound is encoded as a binary indicator.

In [8]:
ac_features['is_DMDBT'] = (ac_features['sulfur_compound'] == 'DMDBT').astype(int)
ac_model_table = ac_features.drop(columns=['sulfur_compound'])
ac_model_table.to_csv('../data/ac_features.csv', index=False)
print(ac_model_table.shape)
ac_model_table

(20, 16)


,material,BET_micro_m2g,BET_meso_m2g,Vmicro_cm3g,Vmeso_cm3g,inorganic_P_wt_pct,inorganic_Ca_wt_pct,inorganic_K_wt_pct,inorganic_Na_wt_pct,carbon_C_wt_pct,carbon_H_wt_pct,carbon_N_wt_pct,oxygen_O_wt_pct,ID_IG_ratio,capacity_mgS_g,is_DMDBT
36,MisAC,563.0,308.0,0.259,0.304,1.0,1.0,3.5,0.2,63.4,1.8,0.2,28.9,1.9,4.70,0
37,MisAC,563.0,308.0,0.259,0.304,1.0,1.0,3.5,0.2,63.4,1.8,0.2,28.9,1.9,9.10,1
38,WAC,634.0,295.0,0.293,0.208,0.1,0.8,1.7,0.0,88.4,1.1,0.1,7.8,1.8,4.20,0
39,WAC,634.0,295.0,0.293,0.208,0.1,0.8,1.7,0.0,88.4,1.1,0.1,7.8,1.8,7.40,1
40,CoAC,614.0,240.0,0.283,0.199,0.3,0.1,2.0,0.3,87.1,1.0,0.5,8.7,2.0,5.10,0
41,CoAC,614.0,240.0,0.283,0.199,0.3,0.1,2.0,0.3,87.1,1.0,0.5,8.7,2.0,7.00,1
42,CAC,351.0,567.0,0.157,0.613,1.7,1.7,0.0,0.6,55.8,2.8,0.2,37.2,1.1,2.30,0
43,CAC,351.0,567.0,0.157,0.613,1.7,1.7,0.0,0.6,55.8,2.8,0.2,37.2,1.1,2.00,1
44,FWAC65,329.0,151.0,0.153,0.163,1.0,0.7,0.8,0.7,75.0,1.5,1.7,18.6,2.1,3.10,0
45,FWAC65,329.0,151.0,0.153,0.163,1.0,0.7,0.8,0.7,75.0,1.5,1.7,18.6,2.1,3.80,1


## Summary

- **Zeolite table**: 16 rows, 9 candidate features (8 independent) (`RE_f_electrons`, `RE_f_dist_from_1`, `Cu_x_f_dist`, `RE_wt_pct`, `Cu_wt_pct`, `Vmicro_cm3g`, `Vmeso_cm3g`, 2 sulfur-compound indicators).
- **Activated-carbon table**: 20 rows, 14 features (`BET_micro_m2g`, `BET_meso_m2g`, `Vmicro_cm3g`, `Vmeso_cm3g`, `inorganic_P_wt_pct`, `inorganic_Ca_wt_pct`, `inorganic_K_wt_pct`, `inorganic_Na_wt_pct`, `carbon_C_wt_pct`, `carbon_H_wt_pct`, `carbon_N_wt_pct`, `oxygen_O_wt_pct`, `ID_IG_ratio`, `is_DMDBT`).